# 05 · Weighted ensemble: GB + RF + XGBoost
Trying fixed weightings of three models. The weights were picked by looking at the validation score, so treat these numbers as slightly optimistic.

In [76]:
import xgboost as xgb
from sklearn.ensemble import GradientBoostingClassifier, RandomForestClassifier
from sklearn.model_selection import train_test_split
from sklearn.metrics import roc_auc_score
import pandas as pd

# Load data
X_train = pd.read_csv("../Data/processed/X_train_clean.csv")
y_train = pd.read_csv("../Data/processed/y_train.csv")['overqualified']

# Split
X_tr, X_val, y_tr, y_val = train_test_split(X_train, y_train, test_size=0.2, random_state=42)

# Train all 3 models
print("Training Gradient Boosting...")
gb = GradientBoostingClassifier(n_estimators=94, random_state=42)
gb.fit(X_tr, y_tr)

print("Training Random Forest...")
rf = RandomForestClassifier(n_estimators=200, random_state=42)
rf.fit(X_tr, y_tr)

print("Training XGBoost...")
xgb_model = xgb.XGBClassifier(n_estimators=150, random_state=42)
xgb_model.fit(X_tr, y_tr)

# Get predictions
pred_gb = gb.predict_proba(X_val)[:, 1]
pred_rf = rf.predict_proba(X_val)[:, 1]
pred_xgb = xgb_model.predict_proba(X_val)[:, 1]

# Try different combinations
combos = [
    (0.6, 0.2, 0.2),
]

print("\nEnsemble Results:")
for w_gb, w_rf, w_xgb in combos:
    ensemble = w_gb * pred_gb + w_rf * pred_rf + w_xgb * pred_xgb
    score = roc_auc_score(y_val, ensemble)
    print(f"GB={w_gb}, RF={w_rf}, XGB={w_xgb}: {score:.4f}")

Training Gradient Boosting...
Training Random Forest...
Training XGBoost...

Ensemble Results:
GB=0.6, RF=0.2, XGB=0.2: 0.7065


In [82]:
combos = [
    (1, 0, 0),
    (0, 1, 0),
    (0, 0, 1),
    (0.5, 0.3, 0.2),  # balanced
    (0.6, 0.2, 0.2),  # favor GB
]
for w_gb, w_rf, w_xgb in combos:
    ensemble = w_gb * pred_gb + w_rf * pred_rf + w_xgb * pred_xgb
    score = roc_auc_score(y_val, ensemble)
    print(f"GB={w_gb}, RF={w_rf}, XGB={w_xgb}: {score:.4f}")


GB=1, RF=0, XGB=0: 0.7052
GB=0, RF=1, XGB=0: 0.6829
GB=0, RF=0, XGB=1: 0.6686
GB=0.5, RF=0.3, XGB=0.2: 0.7054
GB=0.6, RF=0.2, XGB=0.2: 0.7065


In [71]:
from sklearn.ensemble import GradientBoostingClassifier, RandomForestClassifier
from sklearn.model_selection import train_test_split
import xgboost as xgb
from sklearn.metrics import roc_auc_score
import pandas as pd

In [72]:
X_train = pd.read_csv("../Data/processed/X_train_clean.csv")
y_train = pd.read_csv("../Data/processed/y_train.csv")['overqualified']

In [73]:
X_tr, X_val, y_tr, y_val = train_test_split(X_train, y_train, test_size=0.2, random_state=42)